# Entraînement BERT (analyse de sentiment de tweets) sur Colab

Ce notebook entraîne un modèle **BERT** (`bert-base-cased`) à classifier la colonne `text` d'un tweet en 3 classes de sentiment : **negative**, **neutral**, **positive** (colonne `sentiment`).

Il reprend exactement la structure de `train_colab.ipynb` (même `CustomDataset`, même `Model`, mêmes boucles `train_epoch` / `eval_epoch`), adapté au fichier `train-3.csv` (dataset *Tweet Sentiment Extraction*).

In [ ]:
# 1. Vérifier qu'un GPU est bien alloué
import torch
print("CUDA disponible :", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun")

In [ ]:
# 2. Installer les dépendances manquantes (torch/pandas sont déjà présents sur Colab)
%pip install -q transformers wandb tqdm

In [ ]:
# 3. Uploader le CSV depuis ton PC (une fenêtre de sélection de fichier va s'ouvrir)
from google.colab import files

uploaded = files.upload()  # une boîte de dialogue s'ouvre : choisis "train-3.csv" sur ton PC
csv_path = next(iter(uploaded.keys()))
print("Fichier uploadé :", csv_path)

In [ ]:
# 4. (Optionnel) désactiver wandb si tu n'as pas de compte / clé API sous la main
import os
os.environ["WANDB_MODE"] = "disabled"  # commente cette ligne si tu veux logguer sur wandb.ai

In [ ]:
# 5. Dataset + Modèle + boucles d'entraînement (même logique que modeling_bert.py)
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, BertModel
from torch.utils.data import DataLoader, Dataset
import wandb
from tqdm import tqdm


class CustomDataset(Dataset):
    def __init__(self, file_name, tokenizer_name="google-bert/bert-base-cased", max_length=128):
        # encoding="latin-1" car train-3.csv contient des caractères non-UTF-8 (accents, etc.)
        self.df = pd.read_csv(file_name, encoding="latin-1")

        # On ne garde que les colonnes utiles : le texte du tweet et son sentiment
        self.df = self.df[["text", "sentiment"]].dropna()

        # On ne garde que les 3 classes attendues, au cas où
        self.df = self.df[self.df["sentiment"].isin(["negative", "neutral", "positive"])]

        self.text = self.df["text"].astype(str).tolist()
        self.label = self.df["sentiment"].astype(str).tolist()
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

        self.label_id = {"negative": 0, "neutral": 1, "positive": 2}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.text[idx]
        label = self.label[idx]

        ids = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        labels = self.label_id[label]

        return {
            "input_ids": ids["input_ids"].squeeze(0),
            "label": torch.tensor(labels, dtype=torch.long)
        }


class Model(nn.Module):
    def __init__(self, model_name="google-bert/bert-base-cased", num_classes=3):
        super().__init__()
        self.model = BertModel.from_pretrained(model_name)
        self.hidden_dim = self.model.config.hidden_size
        self.proj_lin = nn.Linear(self.hidden_dim, num_classes)

    def forward(self, input_ids):
        x = self.model(input_ids)
        x = x.last_hidden_state[:, 0]
        x = self.proj_lin(x)

        return x


def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(dataloader, total=len(dataloader)):
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def eval_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, total=len(dataloader)):
            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * input_ids.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
# 6. Boucle principale (identique à main() de modeling_bert.py)
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

model_name = "google-bert/bert-base-cased"
batch_size = 32
num_epochs = 3
lr = 2e-5
val_split = 0.1

os.makedirs("checkpoints_bert", exist_ok=True)

wandb.init(
    project="bert-sentiment-classification",
    config={
        "model_name": model_name,
        "batch_size": batch_size,
        "num_epochs": num_epochs,
        "lr": lr,
        "val_split": val_split,
    },
)

dataset = CustomDataset(file_name=csv_path, tokenizer_name=model_name)
print("Nombre d'exemples :", len(dataset))

val_size = int(len(dataset) * val_split)
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

model = Model(model_name=model_name, num_classes=3).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)

    print(f"Epoch {epoch + 1}/{num_epochs} "
          f"| Train loss: {train_loss:.4f}, acc: {train_acc:.4f} "
          f"| Val loss: {val_loss:.4f}, acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train/epoch_loss": train_loss,
        "train/epoch_accuracy": train_acc,
        "val/epoch_loss": val_loss,
        "val/epoch_accuracy": val_acc,
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "./checkpoints_bert/model.pth")
        wandb.run.summary["best_val_accuracy"] = best_val_acc

wandb.finish()

In [ ]:
# 7. Télécharger le checkpoint
# Place ensuite le fichier dans le dossier de ton projet (ex: checkpoints_bert/model.pth)
# pour qu'un éventuel script de démo (demo.py) le retrouve.
from google.colab import files

files.download("checkpoints_bert/model.pth")

## (Bonus) Tester le modèle sur une phrase
Cellule optionnelle pour vérifier rapidement les prédictions du modèle entraîné.

In [ ]:
# 8. (Bonus) Inference rapide sur une phrase de ton choix
model.eval()
id_to_label = {0: "negative", 1: "neutral", 2: "positive"}

def predict_sentiment(text, tokenizer_name="google-bert/bert-base-cased", max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    ids = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    input_ids = ids["input_ids"].to(device)

    with torch.no_grad():
        outputs = model(input_ids)
        pred = outputs.argmax(dim=1).item()

    return id_to_label[pred]

exemple = "I really loved this, it made my day!"
print(f"Texte : {exemple}")
print("Sentiment prédit :", predict_sentiment(exemple))